# 01 — Hyperparameter Tuning

Cari hyperparameter yang lebih baik untuk RandomForest, XGBoost, dan LightGBM
dibanding default `configs/config.yaml`, menggunakan `RandomizedSearchCV`
(scoring `f1_macro` — metrik ini paling sensitif terhadap 35 kelas kontrak
yang timpang, lihat kelas langka 6-18 sampel di train set).

Perbandingan dilakukan di **val set** (bukan test set) supaya test set tetap
bersih untuk evaluasi akhir. Baseline: model dilatih ulang dengan parameter
`configs/config.yaml` saat ini (n_estimators=300 di ketiga model, hasil
retrain 2026-07-09).


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-09" / "outputs" / "hyperparameter_tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_val, y_val = df_val[feature_cols].values, df_val["label"].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, classes: {len(le.classes_)}")
print("Smallest classes in train:")
print(df_train['label'].value_counts().sort_values().head(5))


Train: (7157, 164), Val: (1533, 164), classes: 35
Smallest classes in train:
label
29     5
30     5
32     7
0      7
31    13
Name: count, dtype: int64


## 1. Baseline — parameter saat ini di `configs/config.yaml`

In [2]:
BASELINE_PARAMS = {
    "RandomForest": dict(
        n_estimators=300, max_depth=None, min_samples_split=5,
        random_state=42, n_jobs=-1, class_weight="balanced",
    ),
    "XGBoost": dict(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, eval_metric="mlogloss", verbosity=0,
    ),
    "LightGBM": dict(
        n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbose=-1,
    ),
}

CLASSIFIERS = {
    "RandomForest": RandomForestClassifier,
    "XGBoost": XGBClassifier,
    "LightGBM": LGBMClassifier,
}

baseline_results = []
baseline_models = {}
for name, params in BASELINE_PARAMS.items():
    t0 = time.time()
    clf = CLASSIFIERS[name](**params)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)
    res = evaluate(y_val, y_pred, y_proba, le, model_name=f"{name} (baseline)")
    res["fit_seconds"] = round(time.time() - t0, 1)
    baseline_results.append(res)
    baseline_models[name] = clf
    print_summary(res)

baseline_df = compare_models(baseline_results)
baseline_df



  RandomForest (baseline)
  Accuracy          : 0.4514
  Precision (macro) : 0.2602
  Recall (macro)    : 0.2971
  F1 (macro)        : 0.2693
  F1 (weighted)     : 0.4673
  Top-3 Accuracy    : 0.7123
  Top-5 Accuracy    : 0.8174



  XGBoost (baseline)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM (baseline)
  Accuracy          : 0.5095
  Precision (macro) : 0.3574
  Recall (macro)    : 0.2411
  F1 (macro)        : 0.2613
  F1 (weighted)     : 0.4540
  Top-3 Accuracy    : 0.7469
  Top-5 Accuracy    : 0.8324


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
RandomForest (baseline),0.451402,0.260240,0.297077,0.269270,0.467313,0.712329,0.817352
XGBoost (baseline),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
LightGBM (baseline),0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355


## 2. Ruang pencarian (RandomizedSearchCV)

Dipusatkan di sekitar nilai default saat ini, bukan pencarian buta — supaya
hasil tetap dapat dibandingkan dengan baseline dan waktu run masuk akal untuk
dataset 7.156 baris train.


In [3]:
# NB: XGBoost/LightGBM multiclass boosting builds ~n_estimators * n_classes
# trees per fit (35 classes here) — much more expensive per fit than
# RandomForest. Ranges and n_iter/cv are kept modest so the full sweep
# (RF + XGB + LGBM) finishes in a reasonable time on an 8-core machine.
CV = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
N_ITER = 12
SCORING = "f1_macro"

param_dist_rf = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [None, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5, 0.8],
}

param_dist_xgb = {
    "n_estimators": [150, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

param_dist_lgbm = {
    "n_estimators": [150, 200, 300, 400],
    "num_leaves": [15, 31, 63, 95],
    "max_depth": [-1, 6, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [5, 10, 20, 30],
}

## 3. RandomForest — RandomizedSearchCV

In [4]:
t0 = time.time()
search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1, class_weight="balanced"),
    param_distributions=param_dist_rf,
    n_iter=N_ITER, scoring=SCORING, cv=CV,
    random_state=42, n_jobs=-1, verbose=1,
)
search_rf.fit(X_train, y_train)
print(f"RF search: {time.time() - t0:.1f}s, best CV {SCORING}: {search_rf.best_score_:.4f}")
print(search_rf.best_params_)


Fitting 2 folds for each of 12 candidates, totalling 24 fits


RF search: 80.9s, best CV f1_macro: 0.2490
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}


## 4. XGBoost — RandomizedSearchCV

In [5]:
t0 = time.time()
search_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=1, eval_metric="mlogloss", verbosity=0),
    param_distributions=param_dist_xgb,
    n_iter=N_ITER, scoring=SCORING, cv=CV,
    random_state=42, n_jobs=-1, verbose=1,
)
search_xgb.fit(X_train, y_train)
print(f"XGB search: {time.time() - t0:.1f}s, best CV {SCORING}: {search_xgb.best_score_:.4f}")
print(search_xgb.best_params_)


Fitting 2 folds for each of 12 candidates, totalling 24 fits


XGB search: 172.0s, best CV f1_macro: 0.2466
{'subsample': 1.0, 'reg_lambda': 2.0, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 1.0}


## 5. LightGBM — RandomizedSearchCV

In [6]:
t0 = time.time()
search_lgbm = RandomizedSearchCV(
    LGBMClassifier(random_state=42, n_jobs=1, verbose=-1),
    param_distributions=param_dist_lgbm,
    n_iter=N_ITER, scoring=SCORING, cv=CV,
    random_state=42, n_jobs=-1, verbose=1,
)
search_lgbm.fit(X_train, y_train)
print(f"LGBM search: {time.time() - t0:.1f}s, best CV {SCORING}: {search_lgbm.best_score_:.4f}")
print(search_lgbm.best_params_)


Fitting 2 folds for each of 12 candidates, totalling 24 fits


LGBM search: 163.9s, best CV f1_macro: 0.2453
{'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 1.0}


## 6. Bandingkan tuned vs baseline di val set

In [7]:
searches = {"RandomForest": search_rf, "XGBoost": search_xgb, "LightGBM": search_lgbm}

tuned_results = []
tuned_models = {}
for name, search in searches.items():
    clf = search.best_estimator_
    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)
    res = evaluate(y_val, y_pred, y_proba, le, model_name=f"{name} (tuned)")
    tuned_results.append(res)
    tuned_models[name] = clf
    print_summary(res)

tuned_df = compare_models(tuned_results)

comparison = pd.concat([baseline_df, tuned_df]).sort_index()
comparison



  RandomForest (tuned)
  Accuracy          : 0.4494
  Precision (macro) : 0.2868
  Recall (macro)    : 0.3279
  F1 (macro)        : 0.2937
  F1 (weighted)     : 0.4623
  Top-3 Accuracy    : 0.7221
  Top-5 Accuracy    : 0.8121



  XGBoost (tuned)
  Accuracy          : 0.5212
  Precision (macro) : 0.3302
  Recall (macro)    : 0.2596
  F1 (macro)        : 0.2739
  F1 (weighted)     : 0.4814
  Top-3 Accuracy    : 0.7613
  Top-5 Accuracy    : 0.8408


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM (tuned)
  Accuracy          : 0.5114
  Precision (macro) : 0.2797
  Recall (macro)    : 0.2202
  F1 (macro)        : 0.2324
  F1 (weighted)     : 0.4674
  Top-3 Accuracy    : 0.7528
  Top-5 Accuracy    : 0.8330


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
LightGBM (baseline),0.509459,0.357381,0.241093,0.261329,0.453988,0.746902,0.832355
LightGBM (tuned),0.511416,0.279713,0.220175,0.232407,0.467416,0.752772,0.833007
RandomForest (baseline),0.451402,0.260240,0.297077,0.269270,0.467313,0.712329,0.817352
RandomForest (tuned),0.449446,0.286777,0.327914,0.293680,0.462350,0.722114,0.812133
XGBoost (baseline),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
XGBoost (tuned),0.521200,0.330220,0.259569,0.273884,0.481367,0.761252,0.840835


In [8]:
comparison.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "scoring_used_for_search": SCORING,
    "cv": "StratifiedKFold(n_splits=3)",
    "n_iter": N_ITER,
    "baseline": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in baseline_results},
    "tuned": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in tuned_results},
    "best_params": {name: search.best_params_ for name, search in searches.items()},
    "best_cv_score": {name: float(search.best_score_) for name, search in searches.items()},
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\hyperparameter_tuning\val_comparison.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\hyperparameter_tuning\summary.json


## 7. Kesimpulan

**Diperbarui 2026-07-10 setelah perbaikan split group-aware** (lihat
[README.md](README.md) — split lama bocor lewat pasangan open/closed-room
BBO). Angka di bawah ini dari split yang sudah diperbaiki dan
MENGGANTIKAN kesimpulan sebelumnya.

Hasil di val set (`experiments/2026-07-09/outputs/hyperparameter_tuning/val_comparison.csv`):

| Model | Accuracy (base→tuned) | F1 macro (base→tuned) | F1 weighted (base→tuned) |
|---|---|---|---|
| RandomForest | 45.1% → 44.9% | 0.269 → 0.294 | 0.467 → 0.462 |
| XGBoost | 51.3% → **52.1%** | 0.269 → 0.274 | 0.469 → **0.481** |
| LightGBM | 50.9% → 51.1% | 0.261 → **0.232** | 0.454 → 0.467 |

- **Kesimpulan LightGBM berubah dari versi sebelumnya**: dengan split
  lama (bocor), tuning LightGBM terlihat menaikkan F1 macro (0.304→0.308).
  Dengan split yang diperbaiki, tuning LightGBM justru **menurunkan** F1
  macro (0.261→0.232) — parameter hasil `RandomizedSearchCV` sebelumnya
  ternyata overfit ke pola yang sebagian berasal dari kebocoran
  pasangan open/closed-room, bukan sinyal generalisasi asli.
- **XGBoost tuned tetap yang paling konsisten membaik** — accuracy +0.8pp,
  F1 weighted +0.012, sejalan dengan temuan sebelumnya (walau margin
  sedikit lebih kecil dari klaim awal).
- RandomForest tuned menaikkan F1 macro cukup berarti (0.269→0.294)
  meski accuracy nyaris tidak berubah — trade-off precision/recall,
  bukan perbaikan bersih.
- **Rekomendasi tetap sama**: XGBoost tuned layak divalidasi di test
  set; parameter tuning LightGBM SEBELUMNYA (dari run yang bocor) TIDAK
  boleh dipakai — perlu diulang dari nol jika masih diminati.
- Catatan metodologis dari versi awal (ruang pencarian dipersempit karena
  bug oversubscription CPU) tetap berlaku — celah ini belum ditutup.
